# Evals API：音频输入

本 cookbook 演示如何将 OpenAI 的 Evals 框架用于基于音频的任务。借助 Evals API，我们将对模型针对一条音频消息和提示所生成的回复进行评分：使用**采样（sampling）**生成模型回复，并使用**模型评分（model grading）**根据输出音频和参考答案为模型回复打分。注意，评分针对的是采样回复中的音频输出。

在加入音频支持之前，要评估音频对话，必须先将其转录为文本。现在，你可以直接使用原始音频，并同样以音频形式从模型获取样本。这能更准确地反映真实工作流，例如用户和客服智能体都使用语音的客户支持场景。在评分时，我们将使用一个音频模型，通过模型评分器（model grader）为音频回复打分。我们也可以改用（或同时使用）采样音频的文本转录，并利用现有的一整套文本评分器。

在本示例中，我们将评估模型在以下方面的表现：
1. 针对关于一条音频消息的用户提示，**生成恰当的回复**
2. **与参考答案保持一致**，参考答案代表高质量的回复

## 安装依赖项 + 设置

In [ ]:
# Install required packages
%pip install openai datasets pandas soundfile torch torchcodec pydub jiwer --quiet

In [ ]:
# Import libraries
from datasets import load_dataset, Audio
from openai import OpenAI
import base64
import os
import json
import time
import io
import soundfile as sf
import numpy as np
import pandas as pd

## 数据集准备

我们使用托管在 Hugging Face 上的 [big_bench_audio](https://huggingface.co/datasets/ArtificialAnalysis/big_bench_audio) 数据集。Big Bench Audio 是 Big Bench Hard 部分问题的音频版本。该数据集可用于评估支持音频输入的模型的推理能力。它包含一段描述逻辑问题的音频片段、一个类别以及一个官方答案。

In [ ]:
dataset = load_dataset("ArtificialAnalysis/big_bench_audio")
# Ensure audio column is decoded into a dict with 'array' and 'sampling_rate'
dataset = dataset.cast_column("audio", Audio(decode=True))

我们提取相关字段，并将其整理成类似 JSON 的格式，作为数据源传入 Evals API。输入的音频数据必须是 base64 编码的字符串。我们处理音频文件中的数据并将其转换为 base64。

注意：音频模型目前支持 WAV、MP3、FLAC、Opus 或 PCM16 格式。详情参见[音频输入](https://platform.openai.com/docs/api-reference/chat/create#chat_create-audio)。

In [ ]:
# Audio helpers: file/array to base64
def get_base64(audio_path_or_datauri: str) -> str:
    if audio_path_or_datauri.startswith("data:"):
        # Already base64, just strip prefix
        return audio_path_or_datauri.split(",", 1)[1]
    else:
        # It's a real file path
        with open(audio_path_or_datauri, "rb") as f:
            return base64.b64encode(f.read()).decode("ascii")


def audio_to_base64(audio_val) -> str:
    """
    Accepts various Hugging Face audio representations and returns base64-encoded WAV bytes (no data: prefix).
    Handles:
      - dict or mapping-like with 'path'
      - decoded dict with 'array' and 'sampling_rate'
      - torchcodec AudioDecoder (mapping-like access via ['path'] or ['array'])
      - raw bytes
    """
    # Try to get a file path first
    try:
        path = None
        if isinstance(audio_val, dict) and "path" in audio_val:
            path = audio_val["path"]
        else:
            # Mapping-like access
            try:
                path = audio_val["path"]  # works for many decoder objects
            except Exception:
                path = getattr(audio_val, "path", None)
        if isinstance(path, str) and os.path.exists(path):
            with open(path, "rb") as f:
                return base64.b64encode(f.read()).decode("ascii")
    except Exception:
        pass

    # Fallback: use array + sampling_rate and render to WAV in-memory
    try:
        array = None
        sampling_rate = None
        try:
            array = audio_val["array"]
            sampling_rate = audio_val["sampling_rate"]
        except Exception:
            array = getattr(audio_val, "array", None)
            sampling_rate = getattr(audio_val, "sampling_rate", None)
        if array is not None and sampling_rate is not None:
            audio_np = np.array(array)
            buf = io.BytesIO()
            sf.write(buf, audio_np, int(sampling_rate), format="WAV")
            return base64.b64encode(buf.getvalue()).decode("ascii")
    except Exception:
        pass

    if isinstance(audio_val, (bytes, bytearray)):
        return base64.b64encode(audio_val).decode("ascii")

    raise ValueError("Unsupported audio value; could not convert to base64")


In [ ]:
evals_data_source = []
audio_base64 = None

# Will use the first 3 examples for testing
for example in dataset["train"].select(range(3)):
    audio_val = example["audio"]
    try:
        audio_base64 = audio_to_base64(audio_val)
    except Exception as e:
        print(f"Warning: could not encode audio for id={example['id']}: {e}")
        audio_base64 = None
    evals_data_source.append({
        "item": {
            "id": example["id"],
            "category": example["category"],
            "official_answer": example["official_answer"],
            "audio_base64": audio_base64
        }
    })


如果打印数据源列表，每一项的形式应类似于：

```python
{
  "item": {
    "id": 0
    "category": "formal_fallacies"
    "official_answer": "invalid"
    "audio_base64": "UklGRjrODwBXQVZFZm10IBAAAAABAAEAIlYAAESsA..."
  }
}
```

## 评估配置

现在我们已经有了数据源和任务，接下来创建评估。OpenAI Evals API 的文档请访问 [API 文档](https://platform.openai.com/docs/guides/evals)。


In [ ]:
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

由于音频输入体积较大，我们需要将示例保存到文件中并上传到 API。

In [ ]:
# Save the examples to a file
file_name = "evals_data_source.json"
with open(file_name, "w", encoding="utf-8") as f:
    for obj in evals_data_source:
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

# Upload the file to the API
file = client.files.create(
    file=open(file_name, "rb"),
    purpose="evals"
)

评估由两部分组成："Eval" 和 "Run"。在 "Eval" 中，我们定义数据的预期结构和测试标准（评分器）。

### 数据源配置

根据我们整理好的数据，数据源配置如下：

In [ ]:
data_source_config = {
    "type": "custom",
    "item_schema": {
        "type": "object",
        "properties": {
          "id": { "type": "integer" },
          "category": { "type": "string" },
          "official_answer": { "type": "string" },
          "audio_base64": { "type": "string" }
        },
        "required": ["id", "category", "official_answer", "audio_base64"]
      },
    "include_sample_schema": True, # enables sampling
}

### 测试标准

对于测试标准，我们设置评分器配置。在本示例中，我们使用一个 score_model 评分器，它接收官方答案和采样得到的模型回复（位于 `sample` 命名空间中），然后根据模型回复是否与官方答案一致输出 0 或 1 分。回复中同时包含音频及其文本转录。我们将在评分器中使用音频。关于评分器的更多信息，请访问 [API 评分器文档](https://platform.openai.com/docs/api-reference/graders)。

数据和评分器都设置得当，是有效评估的关键。你很可能需要反复迭代优化评分器的提示。

In [ ]:
grader_config = {
  "type": "score_model",
  "name": "Reference answer audio model grader",
  "model": "gpt-audio",
  "input": [
        {
            "role": "system",
            "content": 'You are a helpful assistant that evaluates audio clips to judge whether they match a provided reference answer. The audio clip is the model''s response to the question. Respond ONLY with a single JSON object matching: {"steps":[{"description":"string","conclusion":"string"}],"result":number}. Do not include any extra text. result must be a float in [0.0, 1.0].'
        },
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": "Evaluate this audio clip to see if it reaches the same conclusion as the reference answer. Reference answer: {{item.official_answer}}",
                },
                {
                    "type": "input_audio",
                    "input_audio": {
                        "data": "{{ sample.output_audio.data }}",
                        "format": "wav",
                    },
                },
            ],
        },
    ],
          "range": [0, 1],
          "pass_threshold": 0.6,
}

另外，我们也可以使用 string_check 评分器，它接收官方答案和采样得到的模型回复（位于 `sample` 命名空间中），然后根据模型回复是否包含参考答案输出 0 到 1 之间的分数。回复中同时包含音频及其文本转录。我们将在评分器中使用文本转录。 

```python
grader_config = {
  "type": "string_check",
  "name": "String check grader",
  "input": "{{sample.output_text}}",
  "reference": "{{item.official_answer}}",
  "operation": "ilike"
}
```

现在，我们创建 eval 对象。

In [ ]:
eval_object = client.evals.create(
        name="Audio Grading Cookbook",
        data_source_config=data_source_config,
        testing_criteria=[grader_config],
    )

## 评估运行

要创建运行，我们需要传入 eval 对象的 id、数据源（即之前整理好的数据），以及用于采样生成模型回复的聊天消息输入。

下面是本示例将使用的采样消息输入。

In [ ]:
sampling_messages = [
    {
        "role": "system",
        "content": "You are a helpful and obedient assistant that can answer questions with audio input. You will be given an audio input containing a question to answer."
    },
    {
        "role": "user",
        "type": "message",
        "content": {
            "type": "input_text",
            "text": "Answer the following question by replying with brief reasoning statements and a conclusion with a single word answer: 'valid' or 'invalid'."
        }
    },
    {
        "role": "user",
        "type": "message",
        "content": {
            "type": "input_audio",
            "input_audio": {
                "data": "{{ item.audio_base64 }}",
                "format": "wav"
            }
        }
    }]

现在启动一次评估运行。

In [ ]:
eval_run = client.evals.runs.create(
        name="Audio Input Eval Run",
        eval_id=eval_object.id,
        data_source={
            "type": "completions", # sample using completions API; responses API is not supported for audio inputs
            "source": {
                "type": "file_id",
                "id": file.id
            },
            "model": "gpt-audio", # model used to generate the response; check that the model you use supports audio inputs
            "sampling_params": {
                "temperature": 0.0,
            },
            "input_messages": {
                "type": "template", 
                "template": sampling_messages},
            "modalities": ["audio", "text"],
        },
    )

## 轮询并显示结果

运行完成后，我们就可以查看结果。你也可以在组织的 OpenAI Evals 仪表板中查看进度和结果。

In [ ]:
while True:
    run = client.evals.runs.retrieve(run_id=eval_run.id, eval_id=eval_object.id)
    if run.status == "completed":
        output_items = list(client.evals.runs.output_items.list(
            run_id=run.id, eval_id=eval_object.id
        ))
        df = pd.DataFrame({
                "id": [item.datasource_item["id"]for item in output_items],
                "category": [item.datasource_item["category"] for item in output_items],
                "official_answer": [item.datasource_item["official_answer"] for item in output_items],
                "model_response": [item.sample.output[0].content for item in output_items],
                "grading_results": ["passed" if item.results[0]["passed"] else "failed"
                                    for item in output_items]
            })
        display(df)
        break
    if run.status == "failed":
        print(run.error)
        break
    time.sleep(5)

### 查看单个输出项

要查看完整的输出项，可以按如下方式操作。输出项的结构在 API 文档的[这里](https://platform.openai.com/docs/api-reference/evals/run-output-item-object)中有说明。

In [ ]:
first_item = output_items[0]

print(json.dumps(dict(first_item), indent=2, default=str))

## 结论

在本 cookbook 中，我们介绍了一个使用 OpenAI Evals API 评估模型原生音频输入的工作流，并演示了如何使用 score model 评分器为音频回复打分。
### 后续步骤
- 将本示例改造为适合你自己的用例。
- 如果你的音频片段较大，可以尝试使用 [uploads API](https://platform.openai.com/docs/api-reference/uploads/create)，它支持最大 8 GB 的文件。
- 前往 [Evals 仪表板](https://platform.openai.com/evaluations)，可视化输出结果并深入了解评估的表现。
